# 01 — Phase 0: Dataset Preparation

**AI Media Authenticity / Deepfake Detection Platform**

This notebook is the entry point for the whole ML pipeline. It:

1. Detects whether we're running on Kaggle or locally
2. Inspects `/kaggle/input/` (if on Kaggle) to find the three mounted datasets
3. Locates each dataset's files on disk (no images/videos are opened yet — paths only)
4. Verifies the files actually exist and reports basic structure/class counts
5. Builds a **balanced 4,000-image subset** (2,000 real + 2,000 fake) for the image pipeline
6. Creates a **stratified 80/10/10 train/val/test split** and saves it to CSV
7. Runs the same discovery pass for the signature and video datasets, honestly reporting
   label reliability instead of assuming folder names mean what we hope they mean

Everything here operates on **filepaths and labels only** — no image, video, or signature
file is loaded into memory in this notebook. That happens lazily, batch-by-batch, in the
model-specific notebooks (02/03/04).

**Datasets used:**
| Modality | Dataset | Kaggle slug |
|---|---|---|
| Image | 140k Real and Fake Faces | `xhlulu/140k-real-and-fake-faces` |
| Signature | Real/Fake Signature Datasets | `emrahaydemr/realfake-signature-datasets` |
| Video | Deep Fake Detection (DFD) Entire Original Dataset | `sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset` |

> If you're running this on Kaggle, add all three datasets to the notebook via
> **"+ Add Input"** before running. If running locally, see `ml/README.md` for the
> Kaggle CLI commands to download small subsets into `ml/datasets/`.


## 1. Imports and project path bootstrap

In [1]:
import sys
import os
from pathlib import Path

import pandas as pd

# --- Locate the ml/ project root regardless of where Jupyter was launched from ---
# We walk upward from the current working directory until we find a folder
# containing 'src/config.py'. This makes the notebook portable across:
#   - Kaggle notebooks (cwd = /kaggle/working)
#   - a local `jupyter lab` launched from ml/ or ml/notebooks/
def find_ml_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(6):
        if (current / "src" / "config.py").exists():
            return current
        current = current.parent
    raise FileNotFoundError(
        "Could not locate the 'ml/' project root (looked for src/config.py "
        "up to 6 parent directories above the current working directory). "
        "If running on Kaggle, add this notebook's 'ml' folder as a utility "
        "script/dataset, or copy ml/src into /kaggle/working/src."
    )

ML_ROOT = find_ml_root(Path.cwd())
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

print(f"ML project root resolved to: {ML_ROOT}")


ML project root resolved to: C:\Users\ADMIN\Downloads\deepfake-detection-platform-phase1-step1\deepfake-detection-platform\ml


In [2]:
from src.utils import set_seed, is_kaggle_env, list_kaggle_inputs
from src import config
from src.data import (
    build_image_metadata, ImageDatasetNotFoundError,
    build_signature_metadata, signature_label_reliability_report, SignatureDatasetNotFoundError,
    build_video_metadata, VideoDatasetNotFoundError,
    build_balanced_subset, stratified_split,
)

set_seed(config.RANDOM_STATE)
pd.set_option("display.max_colwidth", 80)
print("Imports OK. Random seed set to", config.RANDOM_STATE)


Imports OK. Random seed set to 42


## 2. Detect environment and inspect `/kaggle/input/`

In [3]:
print("Running on Kaggle:", is_kaggle_env())

kaggle_inputs = list_kaggle_inputs()
if kaggle_inputs:
    print(f"\nFound {len(kaggle_inputs)} dataset(s) mounted under /kaggle/input:")
    for p in kaggle_inputs:
        print(" -", p.name)
else:
    print("\nNot running on Kaggle (or no datasets mounted yet).")
    print("Falling back to local directories under:", config.DATASETS_ROOT)


Running on Kaggle: False

Not running on Kaggle (or no datasets mounted yet).
Falling back to local directories under: C:\Users\ADMIN\Downloads\deepfake-detection-platform-phase1-step1\deepfake-detection-platform\ml\datasets


## 3. Locate each dataset

`src/config.py` already ran auto-discovery at import time (matching folder-name
keywords against `/kaggle/input/*` first, then falling back to `ml/datasets/<name>/`
locally). We just print what was resolved.


In [ ]:
print("Centralized dataset paths (from src/config.py):\n")
for name, value in config.summary().items():
    print(f"  {name:28s}: {value}")

print("\nRaw path objects:")
print("  IMAGE_DATASET_DIR     :", config.IMAGE_DATASET_DIR)
print("  IMAGE_DATA_DIR        :", config.IMAGE_DATA_DIR)
print("  IMAGE_TRAIN_CSV       :", config.IMAGE_TRAIN_CSV)
print("  IMAGE_VALID_CSV       :", config.IMAGE_VALID_CSV)
print("  IMAGE_TEST_CSV        :", config.IMAGE_TEST_CSV)
print("  SIGNATURE_DATASET_DIR :", config.SIGNATURE_DATASET_DIR)
print("  SIGNATURE_IMAGE_DIR   :", config.SIGNATURE_IMAGE_DIR)
print("  SIGNATURE_METADATA_CSV:", config.SIGNATURE_METADATA_CSV)
print("  VIDEO_DATASET_DIR     :", config.VIDEO_DATASET_DIR)


## 4. Image dataset — load metadata, verify files, show structure

We build metadata directly from the filesystem (walking for `real/` and `fake/`
folders) rather than trusting a specific CSV schema, since dataset mirrors can
differ slightly. This cell will raise a clear, actionable error if the dataset
isn't mounted/downloaded yet — it will NOT silently continue with fake data.


In [ ]:
try:
    image_df = build_image_metadata(config.IMAGE_DATASET_DIR)
    print(f"Found {len(image_df)} image files.")
    print("\nClass distribution (full dataset, before subsetting):")
    print(image_df['label'].value_counts())
    print("\nSample rows:")
    display(image_df.sample(min(5, len(image_df)), random_state=config.RANDOM_STATE))
    IMAGE_DATASET_AVAILABLE = True
except ImageDatasetNotFoundError as e:
    print("IMAGE DATASET NOT AVAILABLE:\n")
    print(e)
    image_df = None
    IMAGE_DATASET_AVAILABLE = False


## 5. Build the balanced image subset (2,000 real + 2,000 fake)

Per the project brief: **do not** train on the full 140k-image dataset. We sample
a fixed, reproducible subset (`random_state=42`) and only ever refer to file paths —
no image is opened or loaded into memory here.


In [ ]:
if IMAGE_DATASET_AVAILABLE:
    image_subset = build_balanced_subset(
        image_df,
        label_col="label",
        n_per_class=config.IMAGE_SUBSET_PER_CLASS,
        random_state=config.RANDOM_STATE,
        classes=config.IMAGE_CLASSES,
    )
    print(f"Balanced subset size: {len(image_subset)} "
          f"(target: {config.IMAGE_SUBSET_PER_CLASS * len(config.IMAGE_CLASSES)})")
    print(image_subset['label'].value_counts())
else:
    image_subset = None
    print("Skipping subset creation - image dataset not available.")


## 6. Stratified 80/10/10 train/val/test split

Expected sizes for a 4,000-image subset: **3,200 train / 400 val / 400 test.**


In [ ]:
if image_subset is not None:
    train_df, val_df, test_df = stratified_split(
        image_subset,
        label_col="label",
        ratios=config.IMAGE_SPLIT_RATIOS,
        random_state=config.RANDOM_STATE,
    )

    print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
    print("\nTrain label balance:\n", train_df['label'].value_counts())
    print("\nVal label balance:\n", val_df['label'].value_counts())
    print("\nTest label balance:\n", test_df['label'].value_counts())
else:
    train_df = val_df = test_df = None


In [ ]:
if train_df is not None:
    config.IMAGE_TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(config.IMAGE_TRAIN_CSV, index=False)
    val_df.to_csv(config.IMAGE_VALID_CSV, index=False)
    test_df.to_csv(config.IMAGE_TEST_CSV, index=False)
    print("Saved:")
    print(" -", config.IMAGE_TRAIN_CSV)
    print(" -", config.IMAGE_VALID_CSV)
    print(" -", config.IMAGE_TEST_CSV)
else:
    print("Nothing to save - image dataset was not available.")


## 7. Signature dataset — discovery + honest label-reliability check

**We do not assume folder/column names like "Gender" or "Age" indicate
genuine/forged/AI-generated.** Instead, `build_signature_metadata` scans each
file's path (relative to the dataset root) for keyword evidence of the three
target classes, and `signature_label_reliability_report` tells us plainly how
much of the dataset got a confident label vs. `"unknown"`.

If reliability is low, **we report that here rather than inventing labels** —
Notebook 03 will only train a classifier on this data if it's reliable enough.


In [ ]:
try:
    signature_df = build_signature_metadata(config.SIGNATURE_DATASET_DIR)
    print(f"Found {len(signature_df)} signature image files.")
    print("\nRaw class distribution (from path-keyword inference):")
    print(signature_df['label'].value_counts())

    reliability = signature_label_reliability_report(signature_df)
    print("\nLabel reliability report:")
    for k, v in reliability.items():
        print(f"  {k}: {v}")

    if not reliability["reliable_enough_for_training"]:
        print(
            "\nWARNING: more than 15% of signature files could not be confidently "
            "labeled from their path. Inspect 'datasets/metadata/signature_metadata.csv' "
            "manually before training a classifier on this data - see the "
            "'matched_keyword' column (empty = unknown) and the actual folder names "
            "in the dataset to determine the real labeling scheme."
        )

    display(signature_df.sample(min(5, len(signature_df)), random_state=config.RANDOM_STATE))
    SIGNATURE_DATASET_AVAILABLE = True
except SignatureDatasetNotFoundError as e:
    print("SIGNATURE DATASET NOT AVAILABLE:\n")
    print(e)
    signature_df = None
    SIGNATURE_DATASET_AVAILABLE = False


In [ ]:
if SIGNATURE_DATASET_AVAILABLE:
    config.SIGNATURE_METADATA_CSV.parent.mkdir(parents=True, exist_ok=True)
    signature_df.to_csv(config.SIGNATURE_METADATA_CSV, index=False)
    print("Saved:", config.SIGNATURE_METADATA_CSV)
else:
    print("Nothing to save - signature dataset was not available.")


## 8. Video dataset — discovery (paths only, no decoding)

We only walk the directory tree for video file paths and infer real/fake from
folder-name keywords - **no video is opened or decoded in this notebook.**
Frame sampling happens lazily in Notebook 04, on a small subset of files.


In [ ]:
try:
    video_df = build_video_metadata(config.VIDEO_DATASET_DIR)
    print(f"Found {len(video_df)} video files.")
    print("\nClass distribution (full dataset, before subsetting):")
    print(video_df['label'].value_counts())
    display(video_df.sample(min(5, len(video_df)), random_state=config.RANDOM_STATE))
    VIDEO_DATASET_AVAILABLE = True
except VideoDatasetNotFoundError as e:
    print("VIDEO DATASET NOT AVAILABLE:\n")
    print(e)
    video_df = None
    VIDEO_DATASET_AVAILABLE = False


In [ ]:
if VIDEO_DATASET_AVAILABLE:
    video_subset = build_balanced_subset(
        video_df,
        label_col="label",
        n_per_class=config.VIDEO_SUBSET_PER_CLASS,
        random_state=config.RANDOM_STATE,
        classes=config.VIDEO_CLASSES,
    )
    video_metadata_csv = config.METADATA_DIR / "video_subset_metadata.csv"
    video_subset.to_csv(video_metadata_csv, index=False)
    print(f"Balanced video subset: {len(video_subset)} files "
          f"(target: {config.VIDEO_SUBSET_PER_CLASS * len(config.VIDEO_CLASSES)})")
    print(video_subset['label'].value_counts())
    print("Saved:", video_metadata_csv)
else:
    video_subset = None
    print("Skipping video subset creation - video dataset not available.")


## 9. Final Phase 0 summary

In [ ]:
print("=" * 60)
print("PHASE 0 - DATASET PREPARATION SUMMARY")
print("=" * 60)

print(f"""
IMAGE
  Available     : {IMAGE_DATASET_AVAILABLE}
  Subset size   : {len(image_subset) if image_subset is not None else 'N/A'}
  Train/Val/Test: {(len(train_df), len(val_df), len(test_df)) if train_df is not None else 'N/A'}

SIGNATURE
  Available             : {SIGNATURE_DATASET_AVAILABLE}
  Total files           : {len(signature_df) if signature_df is not None else 'N/A'}
  Reliable for training : {reliability['reliable_enough_for_training'] if SIGNATURE_DATASET_AVAILABLE else 'N/A'}

VIDEO
  Available   : {VIDEO_DATASET_AVAILABLE}
  Subset size : {len(video_subset) if video_subset is not None else 'N/A'}
""")

print("Next notebook: 02_Image_Deepfake_Detection.ipynb")
